# GNN + LLM Integration: Fixed Experiments

Four ablations on top of the clean GNN-11F baseline:

| # | Variant | What changes |
|---|---------|-------------|
| A | **GNN-11F + LLM-PCA** | PCA-compressed LLM embeddings appended to product features; SAGEConv; no edge weights |
| B | **GNN-11F + LLM-PCA + EdgeWeights** | Same product features; GCNConv replaces SAGEConv on capability edges so cosine weights are applied natively |
| C | **GNN-11F + LLM-PCA + Optuna (GAT)** | PCA features + GATConv + Focal Loss; hyperparameters tuned with Optuna |
| D | **GNN-11F + LLM-PCA + EdgeWeights + Optuna (GAT)** | PCA features + GATConv + cosine edge weights on capability edges + Focal Loss + Optuna |

**Baseline for comparison:** GNN-11F (SAGEConv, 3 product features, binary capability edges, no weights).  
Checkpoint at `data/models/gnn/checkpoints/gnn_11f.pt` — its val PR-AUC is our floor.

In [14]:
import os, sys, pickle, warnings, gc
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)
from sklearn.decomposition import PCA
from sklearn.metrics import precision_recall_curve, auc, roc_auc_score
from torch_geometric.data import HeteroData
from torch_geometric.nn import SAGEConv, GCNConv, GATConv, to_hetero

DATA_DIR  = 'data'
CKPT_DIR  = os.path.join(DATA_DIR, 'models', 'gnn', 'checkpoints')
DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.set_per_process_memory_fraction(0.95)

# Fixed training hyperparameters — same as GNN-11F baseline (for Variants A & B)
HIDDEN   = 128
EPOCHS   = 80
LR       = 1e-3
WD       = 1e-5
PATIENCE = 15
DROP     = 0.3

# Optuna hyperparameter search settings (for Variants C & D)
N_TRIALS     = 40
TRIAL_EPOCHS = 30
TRIAL_PATIENCE = 8

TRAIN_CUTOFF = 2012
VAL_YEAR     = 2013
TEST_YEAR    = 2015

# PCA dimension for LLM embeddings
PCA_DIM = 32

os.makedirs(CKPT_DIR, exist_ok=True)
print(f'Device: {DEVICE}')
print(f'PCA dim: {PCA_DIM}  |  Hidden: {HIDDEN}  |  Epochs: {EPOCHS}')
print(f'Optuna: {N_TRIALS} trials × {TRIAL_EPOCHS} epochs each')

Device: cuda
PCA dim: 32  |  Hidden: 128  |  Epochs: 80
Optuna: 40 trials × 30 epochs each


## Load Artifacts & Build PCA-Compressed Product Features

LLM embeddings are 768-dim unit-normalised vectors. Raw concatenation into product features would dominate the 3 BACI features numerically and add ~700 noisy dimensions. Instead:

1. Fit PCA on all 5018 product embeddings using **training-year** label rows only as the fitting set  
2. Compress to `PCA_DIM=32` — captures the major semantic axes without noise tails  
3. L2-normalise the PCA output so its scale matches the standardised BACI features  
4. Concatenate: `[5018, 3+32] = [5018, 35]` — the new product feature tensor

In [15]:
torch.manual_seed(42); np.random.seed(42)

# ── Standard pipeline artifacts ────────────────────────────────────────────────
edge_idx_raw   = torch.load(os.path.join(DATA_DIR, 'edge_index_by_year.pt'), weights_only=False)
edge_idx_by_yr = {k: v.long() for k, v in edge_idx_raw.items()}
p_x_by_yr_base = torch.load(os.path.join(DATA_DIR, 'product_x_by_year.pt'), weights_only=False)
c_x_11feat     = torch.load(os.path.join(DATA_DIR, 'country_x_by_year.pt'), weights_only=False)

with open(os.path.join(DATA_DIR, 'country_mapping.pkl'), 'rb') as f: c_map = pickle.load(f)
with open(os.path.join(DATA_DIR, 'product_mapping.pkl'), 'rb') as f: p_map = pickle.load(f)

train_lbl = pd.read_csv(os.path.join(DATA_DIR, 'train_labels.csv'))
val_lbl   = pd.read_csv(os.path.join(DATA_DIR, 'val_labels.csv'))
test_lbl  = pd.read_csv(os.path.join(DATA_DIR, 'test_labels.csv'))

# ── LLM embeddings → PCA compression ──────────────────────────────────────────
llm_emb_raw = torch.load(
    os.path.join(DATA_DIR, 'product_llm_embeddings.pt'),
    weights_only=False, map_location='cpu'
).float().numpy()   # [5018, 768], unit-normalised
print(f'Raw LLM embeddings: {llm_emb_raw.shape}')

# Fit PCA on training years only (leakage rule: no test-year info).
# Product embeddings are year-invariant so we just fit on the full 5018-product set —
# but we treat the fitting as a fixed preprocessing step locked to training-era data.
pca = PCA(n_components=PCA_DIM, random_state=42)
pca.fit(llm_emb_raw)

explained = pca.explained_variance_ratio_.sum()
print(f'PCA {PCA_DIM}d explains {explained*100:.1f}% of embedding variance')

# Project all products to PCA space, then L2-normalise per-row so scale ~= BACI features
llm_pca = pca.transform(llm_emb_raw)             # [5018, 32]
row_norms = np.linalg.norm(llm_pca, axis=1, keepdims=True).clip(min=1e-8)
llm_pca_normed = (llm_pca / row_norms).astype(np.float32)
llm_pca_t = torch.from_numpy(llm_pca_normed)     # [5018, 32]

# Augment product features for every year: [5018, 3] → [5018, 3+PCA_DIM]
p_x_with_pca = {}
for yr, base in p_x_by_yr_base.items():
    p_x_with_pca[yr] = torch.cat([base, llm_pca_t], dim=1)

P_IN_BASE = 3
P_IN_PCA  = P_IN_BASE + PCA_DIM   # 35
C_IN      = c_x_11feat[TEST_YEAR].shape[1]   # 11

print(f'\nProduct feature dim:  base={P_IN_BASE}  →  with PCA={P_IN_PCA}')
print(f'Country feature dim:  {C_IN}')
print(f'Sample product[0] features (first 5 of 35): {p_x_with_pca[TEST_YEAR][0, :5].tolist()}')

# ── Capability edge index + cosine weights ─────────────────────────────────────
cap_ei = torch.load(
    os.path.join(DATA_DIR, 'capability_edge_index.pt'), weights_only=False
).long()   # [2, 144192]

# Cosine weights: dot product of unit-normalised embeddings for each edge
llm_unit = torch.from_numpy(llm_emb_raw)   # already unit-normalised
src_np, dst_np = cap_ei[0].numpy(), cap_ei[1].numpy()
cos_weights = torch.tensor(
    (llm_emb_raw[src_np] * llm_emb_raw[dst_np]).sum(axis=1),
    dtype=torch.float32
)   # [144192]  — scalar per edge, range [0.70, 1.0]

print(f'\nCapability edges: {cap_ei.shape}')
print(f'Cosine weight range: [{cos_weights.min():.3f}, {cos_weights.max():.3f}]  mean={cos_weights.mean():.3f}')

Raw LLM embeddings: (5018, 768)
PCA 32d explains 61.6% of embedding variance

Product feature dim:  base=3  →  with PCA=35
Country feature dim:  11
Sample product[0] features (first 5 of 35): [0.4380885362625122, -0.14100655913352966, -0.05737197399139404, 0.1365976482629776, 0.20725679397583008]

Capability edges: torch.Size([2, 144192])
Cosine weight range: [0.319, 1.000]  mean=0.755


## Model Architectures

### Shared components
- `TemporalGNN`: unchanged — GRU over 5-year snapshot sequence  
- `LinkPredictor`: unchanged MLP(256→128→1)

### Variant A — `BipartiteEncoderSAGE_PCA`
- `product_lin: Linear(35, 128)` — projects PCA-augmented features  
- Two SAGEConv layers; capability edges are **unweighted** (topology only)  
- Tests whether LLM semantics in feature space help without any architectural change

### Variant B — `BipartiteEncoderMixed`
- Same PCA-augmented product features  
- **Trade edges** (`exports`, `rev_exports`): SAGEConv — unweighted, appropriate for sparse bipartite  
- **Capability edges** (`capability`): GCNConv — natively accepts `edge_weight`; cosine similarity passed as weight so semantically closer products contribute more  
- Separate conv layers per edge type, merged via `to_hetero`-style manual dispatch

In [16]:
# ── Shared: LinkPredictor (identical to GNN-11F baseline) ─────────────────────
class LinkPredictor(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(hidden * 2, hidden), nn.ReLU(), nn.Dropout(0.2), nn.Linear(hidden, 1))
    def forward(self, zc, zp, ei):
        return self.mlp(torch.cat([zc[ei[0]], zp[ei[1]]], dim=-1)).view(-1)


# ── Shared: TemporalGNN (identical structure, encoder swapped per variant) ─────
class TemporalGNN(nn.Module):
    def __init__(self, enc, hidden):
        super().__init__()
        self.enc   = enc
        self.gru_c = nn.GRU(hidden, hidden)
        self.gru_p = nn.GRU(hidden, hidden)
    def forward(self, snaps):
        cs, ps = [], []
        for s, ew in snaps:   # snaps = list of (HeteroData, edge_weight_tensor|None)
            z = self.enc(s, ew)
            cs.append(z['country']); ps.append(z['product'])
        z_c, _ = self.gru_c(torch.stack(cs))
        z_p, _ = self.gru_p(torch.stack(ps))
        return {'country': z_c[-1], 'product': z_p[-1]}


# ── Variant A: SAGEConv + PCA features, no edge weights ───────────────────────
class _HomoGNN_SAGE(nn.Module):
    """Two-layer SAGEConv. Used with to_hetero — parameter name must be edge_index."""
    def __init__(self, hidden, drop):
        super().__init__()
        self.c1   = SAGEConv(hidden, hidden)
        self.c2   = SAGEConv(hidden, hidden)
        self.drop = drop
    def forward(self, x, edge_index):
        x = F.dropout(self.c1(x, edge_index).relu(), p=self.drop, training=self.training)
        return self.c2(x, edge_index)


class BipartiteEncoderSAGE_PCA(nn.Module):
    """
    Variant A: PCA-augmented product features + SAGEConv.
    capability edges included but unweighted (topology signal only).
    """
    def __init__(self, c_in, p_in, hidden, drop, meta):
        super().__init__()
        self.country_lin = nn.Linear(c_in, hidden)
        self.product_lin = nn.Linear(p_in, hidden)
        self.gnn = to_hetero(_HomoGNN_SAGE(hidden, drop), meta)

    def forward(self, snap, _edge_weight=None):
        x_proj = {
            'country': self.country_lin(snap['country'].x),
            'product': self.product_lin(snap['product'].x),
        }
        return self.gnn(x_proj, snap.edge_index_dict)


# ── Variant B: Mixed SAGEConv + GCNConv with cosine edge weights ───────────────
class MixedBipartiteEncoder(nn.Module):
    """
    Variant B: PCA-augmented product features.
    - Trade edges (exports, rev_exports): SAGEConv  — no natural edge weight
    - Capability edges (product→product):  GCNConv  — cosine weight applied natively

    Manual hetero dispatch: one forward pass calls each edge-type's conv separately,
    then sums neighbour contributions per node type.
    Two layers, ReLU + dropout between.
    """
    def __init__(self, c_in, p_in, hidden, drop):
        super().__init__()
        self.country_lin = nn.Linear(c_in, hidden)
        self.product_lin = nn.Linear(p_in, hidden)
        self.drop = drop

        # Layer 1
        self.sage_exp_1   = SAGEConv(hidden, hidden)   # country→product
        self.sage_rexp_1  = SAGEConv(hidden, hidden)   # product→country
        self.gcn_cap_1    = GCNConv(hidden, hidden, add_self_loops=False, normalize=False)

        # Layer 2
        self.sage_exp_2   = SAGEConv(hidden, hidden)
        self.sage_rexp_2  = SAGEConv(hidden, hidden)
        self.gcn_cap_2    = GCNConv(hidden, hidden, add_self_loops=False, normalize=False)

    def _layer(self, xc, xp, snap, cap_ew, sage_exp, sage_rexp, gcn_cap):
        """One message-passing layer. Returns updated (xc, xp)."""
        ei_exp  = snap['country', 'exports',     'product'].edge_index
        ei_rexp = snap['product', 'rev_exports', 'country'].edge_index
        ei_cap  = snap['product', 'capability',  'product'].edge_index

        # country gets messages from products via rev_exports
        # SAGEConv(src→dst): edge_index[0]=src, edge_index[1]=dst
        xc_new = sage_rexp((xp, xc), ei_rexp)

        # product gets messages from countries via exports
        xp_from_c = sage_exp((xc, xp), ei_exp)

        # product gets weighted messages from products via capability
        xp_from_p = gcn_cap(xp, ei_cap, edge_weight=cap_ew)

        xp_new = xp_from_c + xp_from_p   # additive combination
        return xc_new, xp_new

    def forward(self, snap, cap_ew):
        xc = self.country_lin(snap['country'].x)
        xp = self.product_lin(snap['product'].x)

        # Layer 1
        xc, xp = self._layer(xc, xp, snap, cap_ew,
                              self.sage_exp_1, self.sage_rexp_1, self.gcn_cap_1)
        xc = F.dropout(xc.relu(), p=self.drop, training=self.training)
        xp = F.dropout(xp.relu(), p=self.drop, training=self.training)

        # Layer 2
        xc, xp = self._layer(xc, xp, snap, cap_ew,
                              self.sage_exp_2, self.sage_rexp_2, self.gcn_cap_2)

        return {'country': xc, 'product': xp}


print('Architectures defined.')
print('  TemporalGNN  — shared (snaps = list of (HeteroData, edge_weight|None))')
print('  Variant A    — BipartiteEncoderSAGE_PCA  (SAGEConv + PCA features)')
print('  Variant B    — MixedBipartiteEncoder     (SAGEConv + GCNConv + cosine weights)')

Architectures defined.
  TemporalGNN  — shared (snaps = list of (HeteroData, edge_weight|None))
  Variant A    — BipartiteEncoderSAGE_PCA  (SAGEConv + PCA features)
  Variant B    — MixedBipartiteEncoder     (SAGEConv + GCNConv + cosine weights)


### Variants C & D — GATConv + Focal Loss (Optuna-optimized)

Replace SAGEConv/GCNConv with **GATConv** (attention-based message passing) and **BinaryFocalLoss** (focuses training on hard examples). Hyperparameters — hidden dim, attention heads, dropout, LR, weight decay, focal alpha/gamma — are tuned with **Optuna TPE** over 40 trials.

- **Variant C**: GAT + Focal + PCA features; capability edges unweighted (topology only)
- **Variant D**: GAT + Focal + PCA features; capability edges weighted by cosine similarity via `edge_attr`

In [17]:
# ── Focal Loss ────────────────────────────────────────────────────────────────
class BinaryFocalLoss(nn.Module):
    def __init__(self, alpha: float = 0.5, gamma: float = 2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        bce      = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        p_t      = torch.exp(-bce)
        alpha_t  = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        return (alpha_t * (1 - p_t) ** self.gamma * bce).mean()


# ── GAT block — Variants C & D ───────────────────────────────────────────────
class _GATBlock_PCA(nn.Module):
    """
    Two-layer GATConv.
    edge_dim=1 accepts a single cosine-similarity scalar per capability edge.
    Trade edges pass edge_attr=None; GAT fills via fill_value='mean'.
    """
    def __init__(self, hidden: int, heads: int, drop: float):
        super().__init__()
        self.gat1 = GATConv(hidden, hidden, heads=heads, concat=True,
                             dropout=drop, edge_dim=1, add_self_loops=False, fill_value='mean')
        self.proj = nn.Linear(hidden * heads, hidden)
        self.gat2 = GATConv(hidden, hidden, heads=1, concat=False,
                             dropout=drop, edge_dim=1, add_self_loops=False, fill_value='mean')
        self.drop = drop

    def forward(self, x, edge_index, edge_attr=None):
        x = F.dropout(self.proj(self.gat1(x, edge_index, edge_attr=edge_attr).relu()),
                      p=self.drop, training=self.training)
        return self.gat2(x, edge_index, edge_attr=edge_attr)


class BipartiteEncoderGAT_PCA(nn.Module):
    """Variant C: PCA product features + GATConv via to_hetero (no edge weights)."""
    def __init__(self, c_in: int, p_in: int, hidden: int, heads: int, drop: float, meta):
        super().__init__()
        self.country_lin = nn.Linear(c_in, hidden)
        self.product_lin = nn.Linear(p_in, hidden)
        self.gnn = to_hetero(_GATBlock_PCA(hidden, heads, drop), meta)

    def forward(self, x_dict, ei_dict, ea_dict=None):
        x_proj = {'country': self.country_lin(x_dict['country']),
                  'product': self.product_lin(x_dict['product'])}
        return self.gnn(x_proj, ei_dict, ea_dict) if ea_dict else self.gnn(x_proj, ei_dict)


class TemporalGNN_GAT_PCA(nn.Module):
    """
    Temporal wrapper for GAT-based encoders.
    Accepts list of (HeteroData, ea_dict|None) tuples.
    """
    def __init__(self, enc, hidden):
        super().__init__()
        self.enc   = enc
        self.gru_c = nn.GRU(hidden, hidden)
        self.gru_p = nn.GRU(hidden, hidden)

    def forward(self, snaps):
        cs, ps = [], []
        for s, ea in snaps:
            z = self.enc(s.x_dict, s.edge_index_dict, ea)
            cs.append(z['country']); ps.append(z['product'])
        z_c, _ = self.gru_c(torch.stack(cs))
        z_p, _ = self.gru_p(torch.stack(ps))
        return {'country': z_c[-1], 'product': z_p[-1]}


print('GAT + Focal Loss architectures defined.')
print('  BinaryFocalLoss  — replaces BCEWithLogitsLoss')
print('  BipartiteEncoderGAT_PCA — GATConv + PCA product features')
print('  TemporalGNN_GAT_PCA     — GRU over (HeteroData, ea_dict|None) tuples')

GAT + Focal Loss architectures defined.
  BinaryFocalLoss  — replaces BCEWithLogitsLoss
  BipartiteEncoderGAT_PCA — GATConv + PCA product features
  TemporalGNN_GAT_PCA     — GRU over (HeteroData, ea_dict|None) tuples


## Data Helpers

`build_snap` returns `(HeteroData, cap_ew|None)` — a tuple, never stored on the object.  
This sidesteps PyG's `__getattr__` interception of any attribute ending in `_dict`.

In [18]:
def build_snap(year, p_x_dict, with_cap=False, use_gat=False):
    """
    Returns (HeteroData, extra|None).

    use_gat=False (Variants A & B):
      extra = cos_weights [144192] for GCNConv (Variant B) or None (Variant A)
      Set by with_cap — caller decides which variant consumes it.

    use_gat=True (Variants C & D):
      extra = ea_dict for GATConv: {edge_type: cos_weights_unsqueezed | None}
      capability edges get cos_weights[:,None] as edge_attr; trade edges get None.
    """
    d = HeteroData()
    d['country'].x = c_x_11feat[year]
    d['product'].x = p_x_dict[year]

    ei = edge_idx_by_yr[year].long()
    d['country', 'exports',     'product'].edge_index = ei
    d['product', 'rev_exports', 'country'].edge_index = ei.flip(0)

    extra = None
    if with_cap:
        d['product', 'capability', 'product'].edge_index = cap_ei
        if use_gat:
            # GATConv needs edge_attr=[E,1]; store both on HeteroData and in ea_dict
            d['product', 'capability', 'product'].edge_attr = cos_weights.unsqueeze(1)
            extra = {
                ('country', 'exports',     'product'): None,
                ('product', 'rev_exports', 'country'): None,
                ('product', 'capability',  'product'): cos_weights.unsqueeze(1),
            }
        else:
            extra = cos_weights   # [144192] for GCNConv (Variant B) or ignored (Variant A)
    return d, extra


def build_sample(obs_yr, ldf, p_x_dict, with_cap=False, use_gat=False):
    snaps = [build_snap(y, p_x_dict, with_cap, use_gat) for y in range(obs_yr - 4, obs_yr + 1)]
    row   = ldf[ldf['year'] == obs_yr].copy().reset_index(drop=True)
    ci_s  = row['country'].map(c_map['to_idx'])
    pi_s  = row['product'].map(p_map['to_idx'])
    ok    = ci_s.notna() & pi_s.notna()
    ci    = ci_s[ok].astype(int).values
    pi    = pi_s[ok].astype(int).values
    return {
        'snapshots': snaps,
        'labels': {
            'edge_label_index': torch.tensor([ci, pi], dtype=torch.long),
            'edge_label':       torch.tensor(row.loc[ok, 'label'].values, dtype=torch.float32),
        },
        'year':          int(obs_yr),
        'countries_raw': row.loc[ok, 'country'].values,
        'products_raw':  row.loc[ok, 'product'].values,
    }


def _move_extra(extra, dev):
    if extra is None:
        return None
    if isinstance(extra, dict):
        return {k: (v.to(dev) if v is not None else None) for k, v in extra.items()}
    return extra.to(dev)


def to_dev(samp, dev):
    new_snaps = []
    for d, extra in samp['snapshots']:
        d['country'].x = d['country'].x.to(dev)
        d['product'].x = d['product'].x.to(dev)
        for et in d.edge_types:
            d[et].edge_index = d[et].edge_index.to(device=dev, dtype=torch.long)
            if d[et].get('edge_attr') is not None:
                d[et].edge_attr = d[et].edge_attr.to(dev)
        new_snaps.append((d, _move_extra(extra, dev)))
    samp['snapshots'] = new_snaps
    samp['labels']['edge_label_index'] = samp['labels']['edge_label_index'].to(dev)
    samp['labels']['edge_label']       = samp['labels']['edge_label'].to(dev)


def from_dev(samp):
    new_snaps = []
    for d, extra in samp['snapshots']:
        d['country'].x = d['country'].x.cpu()
        d['product'].x = d['product'].x.cpu()
        for et in d.edge_types:
            d[et].edge_index = d[et].edge_index.cpu()
            if d[et].get('edge_attr') is not None:
                d[et].edge_attr = d[et].edge_attr.cpu()
        new_snaps.append((d, _move_extra(extra, 'cpu')))
    samp['snapshots'] = new_snaps
    samp['labels']['edge_label_index'] = samp['labels']['edge_label_index'].cpu()
    samp['labels']['edge_label']       = samp['labels']['edge_label'].cpu()


@torch.no_grad()
def get_val_prauc(mdl, pred, va, dev):
    mdl.eval(); pred.eval()
    to_dev(va, dev)
    z      = mdl(va['snapshots'])
    scores = torch.sigmoid(
        pred(z['country'], z['product'], va['labels']['edge_label_index'])
    ).cpu().numpy()
    labels = va['labels']['edge_label'].cpu().numpy()
    from_dev(va)
    p, r, _ = precision_recall_curve(labels, scores)
    return float(auc(r, p))


@torch.no_grad()
def get_test_metrics(mdl, pred, te, dev):
    mdl.eval(); pred.eval()
    to_dev(te, dev)
    z      = mdl(te['snapshots'])
    scores = torch.sigmoid(
        pred(z['country'], z['product'], te['labels']['edge_label_index'])
    ).cpu().numpy()
    labels = te['labels']['edge_label'].cpu().numpy()
    from_dev(te)
    p, r, _ = precision_recall_curve(labels, scores)
    prauc   = float(auc(r, p))
    auroc   = float(roc_auc_score(labels, scores))
    return prauc, auroc, scores, labels


# Quick shape check
s0, ew0 = build_snap(TEST_YEAR, p_x_with_pca, with_cap=True, use_gat=False)
print(f'Snap check (SAGE/GCN, with_cap=True):')
print(f'  product.x:  {s0["product"].x.shape}  (expected [5018, {P_IN_PCA}])')
print(f'  cap ew:     {ew0.shape}  range [{ew0.min():.3f}, {ew0.max():.3f}]')

s1, ea1 = build_snap(TEST_YEAR, p_x_with_pca, with_cap=True, use_gat=True)
print(f'\nSnap check (GAT, with_cap=True):')
print(f'  edge_attr shape:  {s1["product","capability","product"].edge_attr.shape}')
print(f'  ea_dict keys:     {list(ea1.keys())}')

Snap check (SAGE/GCN, with_cap=True):
  product.x:  torch.Size([5018, 35])  (expected [5018, 35])
  cap ew:     torch.Size([144192])  range [0.319, 1.000]

Snap check (GAT, with_cap=True):
  edge_attr shape:  torch.Size([144192, 1])
  ea_dict keys:     [('country', 'exports', 'product'), ('product', 'rev_exports', 'country'), ('product', 'capability', 'product')]


## Training Loop

Shared training function. Both variants use:
- `BCEWithLogitsLoss` with positive-class weight (same as GNN-11F baseline)
- Adam, LR=1e-3, WD=1e-5
- Gradient clipping at 1.0
- Early stopping on val PR-AUC, patience=15

In [19]:
def train_variant(name, mdl, pred, tr_samples, va_sample, ckpt_path):
    """
    Train mdl + pred, save best checkpoint.
    Returns (best_val_prauc, mdl, pred) with best weights loaded.
    """
    all_lv = torch.cat([s['labels']['edge_label'] for s in tr_samples])
    n_pos  = all_lv.sum().item()
    n_neg  = (all_lv == 0).sum().item()
    pw     = torch.tensor([n_neg / max(n_pos, 1)], device=DEVICE)
    crit   = nn.BCEWithLogitsLoss(pos_weight=pw)
    opt    = torch.optim.Adam(
        list(mdl.parameters()) + list(pred.parameters()), lr=LR, weight_decay=WD
    )

    best_vpa, no_imp, best_state = -1.0, 0, None
    print(f'\n{"="*60}')
    print(f'Training: {name}')
    print(f'  pos_weight={n_neg/n_pos:.1f}x  |  device={DEVICE}')
    print(f'{"="*60}')

    for ep in range(1, EPOCHS + 1):
        mdl.train(); pred.train()
        ep_loss = 0.0

        for samp in tr_samples:
            to_dev(samp, DEVICE)
            opt.zero_grad()
            z    = mdl(samp['snapshots'])
            loss = crit(
                pred(z['country'], z['product'], samp['labels']['edge_label_index']),
                samp['labels']['edge_label']
            )
            loss.backward()
            nn.utils.clip_grad_norm_(list(mdl.parameters()) + list(pred.parameters()), 1.0)
            opt.step()
            ep_loss += loss.item()
            from_dev(samp)
            del z, loss
            if DEVICE == 'cuda':
                torch.cuda.empty_cache()

        vpa = get_val_prauc(mdl, pred, va_sample, DEVICE)

        if vpa > best_vpa:
            best_vpa  = vpa
            best_state = (
                {k: v.cpu().clone() for k, v in mdl.state_dict().items()},
                {k: v.cpu().clone() for k, v in pred.state_dict().items()},
            )
            no_imp = 0
        else:
            no_imp += 1

        if ep % 10 == 0 or no_imp == 0:
            print(f'  Ep {ep:3d}  loss={ep_loss/len(tr_samples):.4f}  '
                  f'val_PR-AUC={vpa:.4f}  best={best_vpa:.4f}')

        if no_imp >= PATIENCE:
            print(f'  Early stop at epoch {ep}  (best val PR-AUC={best_vpa:.4f})')
            break

    # Restore best weights
    mdl.load_state_dict({k: v.to(DEVICE) for k, v in best_state[0].items()})
    pred.load_state_dict({k: v.to(DEVICE) for k, v in best_state[1].items()})

    # Save checkpoint
    torch.save({
        'mdl_state':      best_state[0],
        'pred_state':     best_state[1],
        'best_val_prauc': best_vpa,
        'pca_dim':        PCA_DIM,
    }, ckpt_path)
    print(f'  Checkpoint saved -> {ckpt_path}  ({os.path.getsize(ckpt_path)/1e6:.1f} MB)')
    return best_vpa, mdl, pred


print('Training loop defined.')

Training loop defined.


## Variant A: GNN-11F + LLM-PCA (SAGEConv, no edge weights)

**What we're testing:** Do PCA-compressed LLM embeddings as product features improve predictions, without changing the GNN architecture?

- Product features: `[5018, 35]` = 3 BACI + 32 PCA dims  
- Architecture: identical to GNN-11F baseline (SAGEConv + to_hetero + GRU)  
- Capability edges included for topology, but unweighted  
- Checkpoint: `gnn_11f_llm_pca.pt`

In [20]:
torch.manual_seed(42); np.random.seed(42)

CKPT_A = os.path.join(CKPT_DIR, 'gnn_11f_llm_pca.pt')

print('Building datasets for Variant A (with_cap=True, PCA product features)...')
tr_A = [build_sample(yr, train_lbl, p_x_with_pca, with_cap=True)
        for yr in sorted(train_lbl['year'].unique())]
va_A = build_sample(VAL_YEAR, val_lbl, p_x_with_pca, with_cap=True)
te_A = build_sample(TEST_YEAR, test_lbl, p_x_with_pca, with_cap=True)
print(f'  Train years: {len(tr_A)}  |  Val pairs: {len(va_A["labels"]["edge_label"]):,}  |  Test pairs: {len(te_A["labels"]["edge_label"]):,}')

# Build model — needs metadata from a snap with capability edges
meta_A = tr_A[0]['snapshots'][0][0].metadata()
enc_A  = BipartiteEncoderSAGE_PCA(C_IN, P_IN_PCA, HIDDEN, DROP, meta_A).to(DEVICE)
mdl_A  = TemporalGNN(enc_A, HIDDEN).to(DEVICE)
pred_A = LinkPredictor(HIDDEN).to(DEVICE)

n_params = sum(p.numel() for p in list(mdl_A.parameters()) + list(pred_A.parameters()))
print(f'  Parameters: {n_params:,}')

vpa_A, mdl_A, pred_A = train_variant('GNN-11F + LLM-PCA', mdl_A, pred_A, tr_A, va_A, CKPT_A)

Building datasets for Variant A (with_cap=True, PCA product features)...
  Train years: 13  |  Val pairs: 128,278  |  Test pairs: 127,531
  Parameters: 434,689

Training: GNN-11F + LLM-PCA
  pos_weight=5.0x  |  device=cuda
  Ep   1  loss=1.0149  val_PR-AUC=0.3347  best=0.3347
  Ep   2  loss=0.9222  val_PR-AUC=0.3488  best=0.3488
  Ep   3  loss=0.9004  val_PR-AUC=0.3594  best=0.3594
  Ep   4  loss=0.8950  val_PR-AUC=0.3658  best=0.3658
  Ep   5  loss=0.8889  val_PR-AUC=0.3733  best=0.3733
  Ep   6  loss=0.8799  val_PR-AUC=0.3782  best=0.3782


KeyboardInterrupt: 

In [ ]:
# ── Variant A test-set evaluation ─────────────────────────────────────────────
prauc_A, auroc_A, scores_A, labels_A = get_test_metrics(mdl_A, pred_A, te_A, DEVICE)

print('\n── Variant A: GNN-11F + LLM-PCA ──')
print(f'  Val  PR-AUC : {vpa_A:.4f}')
print(f'  Test PR-AUC : {prauc_A:.4f}')
print(f'  Test AUROC  : {auroc_A:.4f}')
print()
print('  Interpretation:')
print('  + vs baseline GNN-11F: check test PR-AUC')
print('  PCA features add 32 semantic dims to product representations.')
print('  If PR-AUC improves, LLM-semantic info is genuinely useful in feature space.')


── Variant A: GNN-11F + LLM-PCA ──
  Val  PR-AUC : 0.4676
  Test PR-AUC : 0.4485
  Test AUROC  : 0.8351

  Interpretation:
  + vs baseline GNN-11F: check test PR-AUC
  PCA features add 32 semantic dims to product representations.
  If PR-AUC improves, LLM-semantic info is genuinely useful in feature space.


## Variant B: GNN-11F + LLM-PCA + Edge Weights (SAGEConv + GCNConv)

**What we're testing:** Does weighting capability edges by cosine similarity help, on top of the PCA features?

- Same product features as Variant A: `[5018, 35]`  
- Trade edges: SAGEConv (unweighted — appropriate for sparse bipartite message passing)  
- Capability edges: **GCNConv with `edge_weight=cosine_similarity`** — products that are semantically closer contribute proportionally more  
- Checkpoint: `gnn_11f_llm_pca_ew.pt`

GCNConv formula with weights: `x_i = Σ_j (w_ij / sqrt(deg_i * deg_j)) * W * x_j`  
`normalize=False` is used because we supply pre-computed cosine weights — we don't want GCNConv to re-normalize by degree (which would dilute the cosine signal).

In [21]:
torch.manual_seed(42); np.random.seed(42)

CKPT_B = os.path.join(CKPT_DIR, 'gnn_11f_llm_pca_ew.pt')

# Datasets are the same as Variant A (same PCA features, same capability edges)
# Re-use tr_A / va_A / te_A — they already have cap edges + cos_weights baked in
print('Variant B reuses the same datasets as Variant A.')
print(f'  Capability ew range: [{cos_weights.min():.3f}, {cos_weights.max():.3f}]')

enc_B  = MixedBipartiteEncoder(C_IN, P_IN_PCA, HIDDEN, DROP).to(DEVICE)
mdl_B  = TemporalGNN(enc_B, HIDDEN).to(DEVICE)
pred_B = LinkPredictor(HIDDEN).to(DEVICE)

n_params_B = sum(p.numel() for p in list(mdl_B.parameters()) + list(pred_B.parameters()))
print(f'  Parameters: {n_params_B:,}  (vs Variant A: {n_params:,})')

vpa_B, mdl_B, pred_B = train_variant(
    'GNN-11F + LLM-PCA + EdgeWeights', mdl_B, pred_B, tr_A, va_A, CKPT_B
)

Variant B reuses the same datasets as Variant A.
  Capability ew range: [0.319, 1.000]
  Parameters: 401,921  (vs Variant A: 434,689)


RuntimeError: Expected all tensors to be on the same device, but got tensors is on cuda:0, different from other tensors on cpu (when checking argument in method wrapper_CUDA_cat)

In [ ]:
# ── Variant B test-set evaluation ─────────────────────────────────────────────
prauc_B, auroc_B, scores_B, labels_B = get_test_metrics(mdl_B, pred_B, te_A, DEVICE)

print('\n── Variant B: GNN-11F + LLM-PCA + EdgeWeights ──')
print(f'  Val  PR-AUC : {vpa_B:.4f}')
print(f'  Test PR-AUC : {prauc_B:.4f}')
print(f'  Test AUROC  : {auroc_B:.4f}')


── Variant B: GNN-11F + LLM-PCA + EdgeWeights ──
  Val  PR-AUC : 0.4824
  Test PR-AUC : 0.4565
  Test AUROC  : 0.8392


## Variant C: GNN-11F + LLM-PCA + Optuna (GATConv, no edge weights)

**What we're testing:** Does replacing SAGEConv with attention-based GATConv + Focal Loss + Optuna hyperparameter search improve on the unoptimized PCA variant?

- Same PCA product features as Variant A: `[5018, 35]`  
- GATConv with multi-head attention (heads tuned by Optuna)  
- Focal Loss instead of BCEWithLogitsLoss — focuses on hard, misclassified examples  
- Capability edges included topology-only (no edge weights — just attention over connectivity)  
- Optuna TPE: 40 trials × 30 epochs each; then final full training with best params  
- Checkpoint: `gnn_11f_llm_pca_gat.pt`

In [22]:
torch.manual_seed(42); np.random.seed(42)

CKPT_C = os.path.join(CKPT_DIR, 'gnn_11f_llm_pca_gat.pt')

# Build GAT datasets — capability edges, no edge weights (use_gat=False so no ea_dict)
# Variant C uses GATConv via to_hetero; it doesn't need ea_dict for the unweighted case
print('Building datasets for Variant C (GAT, with_cap=True, PCA features, no edge weights)...')
tr_C = [build_sample(yr, train_lbl, p_x_with_pca, with_cap=True, use_gat=False)
        for yr in sorted(train_lbl['year'].unique())]
va_C = build_sample(VAL_YEAR, val_lbl, p_x_with_pca, with_cap=True, use_gat=False)
te_C = build_sample(TEST_YEAR, test_lbl, p_x_with_pca, with_cap=True, use_gat=False)
print(f'  Train years: {len(tr_C)}  |  Val pairs: {len(va_C["labels"]["edge_label"]):,}')

meta_C = tr_C[0]['snapshots'][0][0].metadata()


def build_gat_model_C(hidden, heads, drop):
    enc  = BipartiteEncoderGAT_PCA(C_IN, P_IN_PCA, hidden, heads, drop, meta_C)
    mdl  = TemporalGNN_GAT_PCA(enc, hidden)
    pred = LinkPredictor(hidden)
    return mdl.to(DEVICE), pred.to(DEVICE)


def run_trial(mdl, pred, crit, opt, tr, va, epochs, patience, trial=None):
    best_vpa, no_imp, best_state = -1.0, 0, None
    for ep in range(1, epochs + 1):
        mdl.train(); pred.train()
        for samp in tr:
            to_dev(samp, DEVICE)
            opt.zero_grad()
            z    = mdl(samp['snapshots'])
            loss = crit(pred(z['country'], z['product'],
                             samp['labels']['edge_label_index']),
                        samp['labels']['edge_label'])
            loss.backward()
            nn.utils.clip_grad_norm_(list(mdl.parameters()) + list(pred.parameters()), 1.0)
            opt.step()
            from_dev(samp)
            del z, loss
            if DEVICE == 'cuda': torch.cuda.empty_cache()
        vpa = get_val_prauc(mdl, pred, va, DEVICE)
        if vpa > best_vpa:
            best_vpa = vpa
            best_state = ({k: v.cpu().clone() for k, v in mdl.state_dict().items()},
                          {k: v.cpu().clone() for k, v in pred.state_dict().items()})
            no_imp = 0
        else:
            no_imp += 1
        if trial is not None:
            trial.report(vpa, ep)
            if trial.should_prune():
                raise optuna.TrialPruned()
        if no_imp >= patience:
            break
    return best_vpa, best_state


def objective_C(trial):
    hidden  = trial.suggest_categorical('hidden_dim', [64, 128])
    heads   = trial.suggest_categorical('gat_heads',  [2, 4])
    drop    = trial.suggest_float('dropout',          0.1, 0.5)
    lr      = trial.suggest_float('lr',               1e-4, 1e-2, log=True)
    wd      = trial.suggest_float('weight_decay',     1e-6, 1e-3, log=True)
    f_alpha = trial.suggest_float('focal_alpha',      0.25, 0.85)
    f_gamma = trial.suggest_float('focal_gamma',      1.0, 3.0)

    torch.manual_seed(42)
    mdl, pred = build_gat_model_C(hidden, heads, drop)
    crit = BinaryFocalLoss(alpha=f_alpha, gamma=f_gamma)
    opt  = torch.optim.Adam(list(mdl.parameters()) + list(pred.parameters()),
                             lr=lr, weight_decay=wd)
    best_vpa, _ = run_trial(mdl, pred, crit, opt, tr_C, va_C,
                             TRIAL_EPOCHS, TRIAL_PATIENCE, trial)
    del mdl, pred, crit, opt
    if DEVICE == 'cuda': torch.cuda.empty_cache()
    gc.collect()
    return best_vpa


print(f'Starting Optuna (Variant C): {N_TRIALS} trials × {TRIAL_EPOCHS} epochs...')
study_C = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=5),
)
study_C.optimize(objective_C, n_trials=N_TRIALS, show_progress_bar=True)

bp_C = study_C.best_trial.params
print(f'\nBest trial #{study_C.best_trial.number}  val PR-AUC = {study_C.best_trial.value:.4f}')
for k, v in bp_C.items():
    print(f'  {k}: {v}')

Building datasets for Variant C (GAT, with_cap=True, PCA features, no edge weights)...
  Train years: 13  |  Val pairs: 128,278
Starting Optuna (Variant C): 40 trials × 30 epochs...


  0%|          | 0/40 [00:00<?, ?it/s]

[W 2026-06-12 19:06:14,012] Trial 0 failed with parameters: {'hidden_dim': 128, 'gat_heads': 2, 'dropout': 0.1624074561769746, 'lr': 0.00020511104188433984, 'weight_decay': 1.493656855461763e-06, 'focal_alpha': 0.769705687464961, 'focal_gamma': 2.2022300234864174} because of the following error: RuntimeError('Boolean value of Tensor with more than one value is ambiguous').
Traceback (most recent call last):
  File "c:\Users\Ashwa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\optuna\study\_optimize.py", line 205, in _run_trial
    value_or_values = func(trial)
  File "C:\Users\Ashwa\AppData\Local\Temp\ipykernel_5760\963990315.py", line 72, in objective_C
    best_vpa, _ = run_trial(mdl, pred, crit, opt, tr_C, va_C,
                  ~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
                             TRIAL_EPOCHS, TRIAL_PATIENCE, trial)
                             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Ashwa\AppData\Local\Temp\ipykernel_5760\963990315.py"

RuntimeError: Boolean value of Tensor with more than one value is ambiguous

In [23]:
torch.manual_seed(42); np.random.seed(42)

print('Final training (Variant C) with best hyperparameters:')
print(f'  hidden={bp_C["hidden_dim"]}  heads={bp_C["gat_heads"]}  dropout={bp_C["dropout"]:.3f}')
print(f'  lr={bp_C["lr"]:.2e}  wd={bp_C["weight_decay"]:.2e}')
print(f'  focal_alpha={bp_C["focal_alpha"]:.3f}  focal_gamma={bp_C["focal_gamma"]:.3f}')

mdl_C, pred_C = build_gat_model_C(bp_C['hidden_dim'], bp_C['gat_heads'], bp_C['dropout'])
crit_C = BinaryFocalLoss(alpha=bp_C['focal_alpha'], gamma=bp_C['focal_gamma'])
opt_C  = torch.optim.Adam(list(mdl_C.parameters()) + list(pred_C.parameters()),
                           lr=bp_C['lr'], weight_decay=bp_C['weight_decay'])

best_vpa_C, best_state_C = run_trial(
    mdl_C, pred_C, crit_C, opt_C, tr_C, va_C, EPOCHS, PATIENCE
)
print(f'  Best val PR-AUC: {best_vpa_C:.4f}')

# Restore and save
mdl_C.load_state_dict({k: v.to(DEVICE) for k, v in best_state_C[0].items()})
pred_C.load_state_dict({k: v.to(DEVICE) for k, v in best_state_C[1].items()})

torch.save({
    'mdl_state':      best_state_C[0],
    'pred_state':     best_state_C[1],
    'best_val_prauc': best_vpa_C,
    'c_in':      C_IN,
    'p_in':      P_IN_PCA,
    'hidden':    bp_C['hidden_dim'],
    'heads':     bp_C['gat_heads'],
    'dropout':   bp_C['dropout'],
    'pca_dim':   PCA_DIM,
    'focal_alpha': bp_C['focal_alpha'],
    'focal_gamma': bp_C['focal_gamma'],
    'best_hparams': dict(bp_C),
}, CKPT_C)
print(f'  Checkpoint saved -> {CKPT_C}  ({os.path.getsize(CKPT_C)/1e6:.1f} MB)')

prauc_C, auroc_C, scores_C, labels_C = get_test_metrics(mdl_C, pred_C, te_C, DEVICE)
print(f'\n── Variant C: GNN-11F + LLM-PCA + GAT (Optuna) ──')
print(f'  Val  PR-AUC : {best_vpa_C:.4f}')
print(f'  Test PR-AUC : {prauc_C:.4f}')
print(f'  Test AUROC  : {auroc_C:.4f}')

Final training (Variant C) with best hyperparameters:


NameError: name 'bp_C' is not defined

## Variant D: GNN-11F + LLM-PCA + Edge Weights + Optuna (GATConv)

**What we're testing:** Does adding cosine similarity as edge attributes on top of GAT + Focal + Optuna further improve over Variant C?

- Same PCA product features: `[5018, 35]`  
- GATConv with `edge_dim=1` — cosine similarity passed as `edge_attr` on capability edges  
- GAT attention scores are modulated by both node embeddings AND the cosine weight  
- Focal Loss + Optuna (same search space as Variant C)  
- Checkpoint: `gnn_11f_llm_pca_gat_ew.pt`

This is the most fully integrated variant: semantic features (PCA) + semantic topology (capability edges) + semantic edge weights (cosine via GAT attention) + adaptive loss (Focal) + tuned hyperparameters (Optuna).

In [ ]:
torch.manual_seed(42); np.random.seed(42)

CKPT_D = os.path.join(CKPT_DIR, 'gnn_11f_llm_pca_gat_ew.pt')

# GAT + cosine edge weights — use_gat=True so build_snap returns ea_dict
print('Building datasets for Variant D (GAT + cosine edge_attr, with_cap=True)...')
tr_D = [build_sample(yr, train_lbl, p_x_with_pca, with_cap=True, use_gat=True)
        for yr in sorted(train_lbl['year'].unique())]
va_D = build_sample(VAL_YEAR, val_lbl, p_x_with_pca, with_cap=True, use_gat=True)
te_D = build_sample(TEST_YEAR, test_lbl, p_x_with_pca, with_cap=True, use_gat=True)
print(f'  Train years: {len(tr_D)}  |  Val pairs: {len(va_D["labels"]["edge_label"]):,}')

meta_D = tr_D[0]['snapshots'][0][0].metadata()


def build_gat_model_D(hidden, heads, drop):
    enc  = BipartiteEncoderGAT_PCA(C_IN, P_IN_PCA, hidden, heads, drop, meta_D)
    mdl  = TemporalGNN_GAT_PCA(enc, hidden)
    pred = LinkPredictor(hidden)
    return mdl.to(DEVICE), pred.to(DEVICE)


def objective_D(trial):
    hidden  = trial.suggest_categorical('hidden_dim', [64, 128])
    heads   = trial.suggest_categorical('gat_heads',  [2, 4])
    drop    = trial.suggest_float('dropout',          0.1, 0.5)
    lr      = trial.suggest_float('lr',               1e-4, 1e-2, log=True)
    wd      = trial.suggest_float('weight_decay',     1e-6, 1e-3, log=True)
    f_alpha = trial.suggest_float('focal_alpha',      0.25, 0.85)
    f_gamma = trial.suggest_float('focal_gamma',      1.0, 3.0)

    torch.manual_seed(42)
    mdl, pred = build_gat_model_D(hidden, heads, drop)
    crit = BinaryFocalLoss(alpha=f_alpha, gamma=f_gamma)
    opt  = torch.optim.Adam(list(mdl.parameters()) + list(pred.parameters()),
                             lr=lr, weight_decay=wd)
    best_vpa, _ = run_trial(mdl, pred, crit, opt, tr_D, va_D,
                             TRIAL_EPOCHS, TRIAL_PATIENCE, trial)
    del mdl, pred, crit, opt
    if DEVICE == 'cuda': torch.cuda.empty_cache()
    gc.collect()
    return best_vpa


print(f'Starting Optuna (Variant D): {N_TRIALS} trials × {TRIAL_EPOCHS} epochs...')
study_D = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=5),
)
study_D.optimize(objective_D, n_trials=N_TRIALS, show_progress_bar=True)

bp_D = study_D.best_trial.params
print(f'\nBest trial #{study_D.best_trial.number}  val PR-AUC = {study_D.best_trial.value:.4f}')
for k, v in bp_D.items():
    print(f'  {k}: {v}')

Building datasets for Variant D (GAT + cosine edge_attr, with_cap=True)...
  Train years: 13  |  Val pairs: 128,278
Starting Optuna (Variant D): 40 trials × 30 epochs...


  0%|          | 0/40 [00:00<?, ?it/s]

In [ ]:
torch.manual_seed(42); np.random.seed(42)

print('Final training (Variant D) with best hyperparameters:')
print(f'  hidden={bp_D["hidden_dim"]}  heads={bp_D["gat_heads"]}  dropout={bp_D["dropout"]:.3f}')
print(f'  lr={bp_D["lr"]:.2e}  wd={bp_D["weight_decay"]:.2e}')
print(f'  focal_alpha={bp_D["focal_alpha"]:.3f}  focal_gamma={bp_D["focal_gamma"]:.3f}')

mdl_D, pred_D = build_gat_model_D(bp_D['hidden_dim'], bp_D['gat_heads'], bp_D['dropout'])
crit_D = BinaryFocalLoss(alpha=bp_D['focal_alpha'], gamma=bp_D['focal_gamma'])
opt_D  = torch.optim.Adam(list(mdl_D.parameters()) + list(pred_D.parameters()),
                           lr=bp_D['lr'], weight_decay=bp_D['weight_decay'])

best_vpa_D, best_state_D = run_trial(
    mdl_D, pred_D, crit_D, opt_D, tr_D, va_D, EPOCHS, PATIENCE
)
print(f'  Best val PR-AUC: {best_vpa_D:.4f}')

# Restore and save
mdl_D.load_state_dict({k: v.to(DEVICE) for k, v in best_state_D[0].items()})
pred_D.load_state_dict({k: v.to(DEVICE) for k, v in best_state_D[1].items()})

torch.save({
    'mdl_state':      best_state_D[0],
    'pred_state':     best_state_D[1],
    'best_val_prauc': best_vpa_D,
    'c_in':      C_IN,
    'p_in':      P_IN_PCA,
    'hidden':    bp_D['hidden_dim'],
    'heads':     bp_D['gat_heads'],
    'dropout':   bp_D['dropout'],
    'pca_dim':   PCA_DIM,
    'focal_alpha': bp_D['focal_alpha'],
    'focal_gamma': bp_D['focal_gamma'],
    'best_hparams': dict(bp_D),
}, CKPT_D)
print(f'  Checkpoint saved -> {CKPT_D}  ({os.path.getsize(CKPT_D)/1e6:.1f} MB)')

prauc_D, auroc_D, scores_D, labels_D = get_test_metrics(mdl_D, pred_D, te_D, DEVICE)
print(f'\n── Variant D: GNN-11F + LLM-PCA + GAT + EdgeWeights (Optuna) ──')
print(f'  Val  PR-AUC : {best_vpa_D:.4f}')
print(f'  Test PR-AUC : {prauc_D:.4f}')
print(f'  Test AUROC  : {auroc_D:.4f}')

## Results Summary

In [ ]:
baseline_val_prauc = None
baseline_ckpt = os.path.join(CKPT_DIR, 'gnn_11f.pt')
if os.path.exists(baseline_ckpt):
    ck = torch.load(baseline_ckpt, weights_only=False, map_location='cpu')
    baseline_val_prauc = ck.get('best_val_prauc', None)

print('╔══════════════════════════════════════════════════════════════════════════╗')
print('║           LLM Integration Results — Test Year 2015                      ║')
print('╠══════════════════════════════════════════════════════════════════════════╣')
print(f'║  {"Method":<44} {"Val PR-AUC":>10} {"Test PR-AUC":>11} {"AUROC":>7} ║')
print('╠══════════════════════════════════════════════════════════════════════════╣')

results = [
    ('GNN-11F baseline (SAGEConv, 3 feat)',         baseline_val_prauc, None,     None),
    ('Variant A: + LLM-PCA (SAGE)',                 vpa_A,             prauc_A,  auroc_A),
    ('Variant B: + LLM-PCA + cosine ew (GCN)',      vpa_B,             prauc_B,  auroc_B),
    ('Variant C: + LLM-PCA + GAT Optuna',           best_vpa_C,        prauc_C,  auroc_C),
    ('Variant D: + LLM-PCA + GAT + ew Optuna',      best_vpa_D,        prauc_D,  auroc_D),
]

for name, vpa, tpa, auroc in results:
    vpa_s   = f'{vpa:.4f}' if vpa is not None else '(no ckpt)'
    tpa_s   = f'{tpa:.4f}' if tpa is not None else '[run eval]'
    auroc_s = f'{auroc:.4f}' if auroc is not None else '[run eval]'
    print(f'║  {name:<44} {vpa_s:>10} {tpa_s:>11} {auroc_s:>7} ║')

print('╚══════════════════════════════════════════════════════════════════════════╝')

print(f'\nDeltas (Test PR-AUC):')
print(f'  B vs A  (edge weights on SAGE/GCN):            {prauc_B - prauc_A:+.4f}')
print(f'  C vs A  (GAT+Focal+Optuna vs plain SAGE):      {prauc_C - prauc_A:+.4f}')
print(f'  D vs C  (cosine edge_attr on GAT):             {prauc_D - prauc_C:+.4f}')
print(f'  D vs B  (optimized vs unoptimized ew):         {prauc_D - prauc_B:+.4f}')

print(f'\nCheckpoints:')
for ck in [CKPT_A, CKPT_B, CKPT_C, CKPT_D]:
    size_mb = os.path.getsize(ck) / 1e6 if os.path.exists(ck) else 0
    print(f'  {ck}  ({size_mb:.1f} MB)')